<a href="https://colab.research.google.com/github/bcalm-2/gen-ai-assignments-3.4.5/blob/main/Srashti_GenAiAssignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ACTIVITY 1: PROMPT ENGINEERING

!pip install transformers -q

from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load model
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Zero-Shot
input_text = "question: What is Artificial Intelligence? answer:"
inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=200,
    min_length=50,
    num_beams=4,
    no_repeat_ngram_size=2
)

print("Zero-shot Output:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


# Few-Shot
input_text = """
question: What is AI? answer: AI is intelligence shown by machines.
question: What is Machine Learning? answer: Machine learning learns from data.
question: What is Deep Learning? answer:
"""

inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=200,
    min_length=50,
    num_beams=4,
    no_repeat_ngram_size=2
)

print("\nFew-shot Output:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


# Chain-of-Thought
input_text = "question: What is Machine Learning? answer: Explain step by step."
inputs = tokenizer(input_text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_length=250,
    min_length=70,
    num_beams=4,
    no_repeat_ngram_size=2
)

print("\nCoT Output:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# ACTIVITY 2: LoRA FINE-TUNING

!pip install datasets peft accelerate torch -q

from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments

# Create Dataset
data = {
    "input": [
        "question: What is AI?",
        "question: What is Machine Learning?"
    ],
    "output": [
        "AI is the simulation of human intelligence in machines.",
        "Machine Learning is a subset of AI that learns from data."
    ]
}

dataset = Dataset.from_dict(data)

# Preprocessing
def preprocess(example):
    inputs = tokenizer(example["input"], padding="max_length", truncation=True)
    labels = tokenizer(example["output"], padding="max_length", truncation=True)
    inputs["labels"] = labels["input_ids"]
    return inputs

dataset = dataset.map(preprocess)

# LoRA Configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.1
)

# Apply LoRA
lora_model = get_peft_model(model, lora_config)

# Training Setup
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    logging_steps=1
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=dataset
)

# Train Model
trainer.train()

# Save Model
lora_model.save_pretrained("lora-t5-model")

In [ ]:
# ACTIVITY 3: PERFORMANCE COMPARISON

# Test Input
test_input = "question: What is AI? answer:"
inputs = tokenizer(test_input, return_tensors="pt")

# Base Model Output
base_output = model.generate(
    **inputs,
    max_length=200,
    min_length=50,
    num_beams=4,
    no_repeat_ngram_size=2
)

print("Base Model Output:")
print(tokenizer.decode(base_output[0], skip_special_tokens=True))

# LoRA Model Output
lora_output = lora_model.generate(
    **inputs,
    max_length=200,
    min_length=50,
    num_beams=4,
    no_repeat_ngram_size=2
)

print("\nLoRA Fine-Tuned Model Output:")
print(tokenizer.decode(lora_output[0], skip_special_tokens=True))